# 0 · Setup

Run this notebook once before any of the analysis notebooks. It downloads the IQB data to the `data` folder for the specified date range, then builds the country context table we use to group and validate the scores. Downloads are cached, so re-running it only fetches files that are missing.

## IQB Data

We use M-Lab's NDT percentile tables for the 24 months from Jan 2024 to Dec 2025 (`data/v1/`), taken from M-Lab's IQB data manifest. Each row is one place-month, with its test counts and the nine reported percentiles (P1, P5, P10, P25, P50, P75, P90, P95, P99) of download, upload, latency and packet loss.

We download four table types, each for download and upload tests:

| Table | One row per | Used by |
|---|---|---|
| `country` | country-month | most notebooks |
| `country_subdivision1` | region-month | 5-map |
| `country_city` | city-month | 4a-urban-rural |
| `country_asn` | network-month (autonomous system) | the test-share HHI below |

M-Lab also publishes regions and cities split by network (`country_subdivision1_asn`, `country_city_asn`, about 2 GB together). No notebook uses them, so we do not download them.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(f"Python 3.12+ required, found {sys.version.split()[0]}")
print("Python version OK")

import hashlib
import json
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import requests
from scipy.stats import spearmanr

from iqb import IQBCalculator  # not used here; fails early if the IQB library is missing
import iqb_tools as T

T.set_style()
pd.set_option("display.width", 140, "display.max_columns", 30)

DATA_CACHE = T.ROOT / "data"  # T.load_level reads the tables from data/v1/ under the repository root
DATA_CACHE.mkdir(parents=True, exist_ok=True)
print(f"Root dir: {T.ROOT}")
print(f"Data target: {DATA_CACHE}")

### Download the M-Lab tables

In [ ]:
PULL_START = pd.Timestamp("2024-01-01")
PULL_END = pd.Timestamp("2025-12-01")  # first day of the last month to download (inclusive)
TABLES = {"country", "country_asn", "country_city", "country_subdivision1"}
MANIFEST_URL = (
    "https://raw.githubusercontent.com/m-lab/iqb/main/data/state/ghremote/manifest.json"
)

r = requests.get(MANIFEST_URL, timeout=60)
r.raise_for_status()
manifest = r.json()


def data(path):
    # cache/v1/<month start>/<month end>/<downloads|uploads>_by_<table>/data.parquet
    parts = Path(path).parts
    return (
        path.endswith(".parquet")
        and parts[4].split("_by_", 1)[1] in TABLES
        and PULL_START <= pd.Timestamp(parts[2][:8]) <= PULL_END
    )


files = {path: meta for path, meta in manifest["files"].items() if data(path)}
todo = {
    path: meta
    for path, meta in files.items()
    if not (DATA_CACHE / Path(*Path(path).parts[1:])).exists()
}
print(f"{len(files) - len(todo)} / {len(files)} files already cached, downloading {len(todo)}")

for i, (path, meta) in enumerate(todo.items(), 1):
    dest = DATA_CACHE / Path(*Path(path).parts[1:])  # strip leading 'cache/'
    dest.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(meta["url"], timeout=300)
    r.raise_for_status()
    if hashlib.sha256(r.content).hexdigest() != meta["sha256"]:
        raise ValueError(f"Checksum mismatch: {path}")
    tmp = dest.with_suffix(".part")
    tmp.write_bytes(r.content)
    tmp.replace(dest)  # never leaves a half-written file behind
    if i % 5 == 0:
        print(f"  Downloaded {i} / {len(todo)}")


In [ ]:
df = T.load_level("country")
print(
    f"{len(df):,} country-months, {df.country_code.nunique()} countries, "
    f"{df.month.min():%Y-%m} to {df.month.max():%Y-%m}"
)

cov = df.groupby(df.month.dt.year).agg(
    countries=("country_code", "nunique"),
    tests_down=("n_down", "sum"),
    tests_up=("n_up", "sum"),
)
cov["median_tests_per_country_month"] = df.groupby(df.month.dt.year).n_down.median()
cov


## Development context, ITU indicators and ISP market concentration

We assemble the external variables the proposal uses to stratify and validate IQB: the World Bank income classification, gross domestic product (GDP) per capita, International Telecommunication Union (ITU) connectivity indicators and Internet service provider (ISP) market concentration, measured by the Herfindahl–Hirschman Index (HHI).

| Variable | Source | Notes |
|---|---|---|
| Income group, region | World Bank country API (current classification) | 4 income groups; "Not classified" kept as missing |
| GDP per capita, PPP (constant 2021 int. $) | WDI `NY.GDP.PCAP.PP.KD` | latest value 2021–2025 |
| Fixed broadband subscriptions per 100 | ITU, via WDI `IT.NET.BBND.P2` | ITU's own API refuses scripted access, so ITU series are taken from their WDI republication |
| Internet users, % of population | ITU, via WDI `IT.NET.USER.ZS` | |
| Mobile subscriptions per 100 | ITU, via WDI `IT.CEL.SETS.P2` | |
| Secure Internet servers per million | Netcraft, via WDI `IT.NET.SECR.P6` | an infrastructure-maturity proxy |
| Urban population, % | WDI `SP.URB.TOTL.IN.ZS` | used in 4a-urban-rural |
| ISP HHI | APNIC Labs, estimated Internet users per AS | market shares of access networks, grouped by operator |
| M-Lab test-share HHI | M-Lab NDT tests per AS, 2025 | for comparison only (shares of *tests*, not users) |

*Why APNIC for HHI.* Subscriber market shares are not openly available for most countries, so we use APNIC Labs' estimates instead. APNIC estimates the number of Internet users served by each autonomous system (AS) in each country from a large ad-based measurement, a standard open proxy for access-market shares. M-Lab's own test counts per AS measure who runs speed tests, which over-represents some networks. We compute both and compare them.

*Caveats.* APNIC shares cover fixed and mobile access together and refer to the latest 60-day window. Market structure changes slowly, so we treat this as acceptable for a cross-section. Several ASes can belong to one operator, so we group ASes by the operator name APNIC reports.


In [ ]:
WB = T.EXT / "worldbank"
WB.mkdir(parents=True, exist_ok=True)
AP = T.EXT / "apnic"
AP.mkdir(parents=True, exist_ok=True)


def cached_json(url, path, params=None):
    """Download once, then reuse the raw response from disk."""
    if not path.exists():
        r = requests.get(url, params=params, timeout=180)
        r.raise_for_status()
        path.write_text(r.text)
        time.sleep(0.2)
    return json.loads(path.read_text())


## World Bank country metadata and income groups

In [ ]:
meta = cached_json(
    "https://api.worldbank.org/v2/country",
    WB / "countries.json",
    {"format": "json", "per_page": 400},
)[1]
countries = pd.DataFrame(
    {
        "iso3": [c["id"] for c in meta],
        "country_code": [c["iso2Code"] for c in meta],
        "wb_name": [c["name"] for c in meta],
        "region": [c["region"]["value"].strip() for c in meta],
        "income_group": [c["incomeLevel"]["value"] for c in meta],
    }
)
countries = countries[countries.region != "Aggregates"].copy()
countries.loc[~countries.income_group.isin(T.INCOME_ORDER), "income_group"] = np.nan
print(
    len(countries),
    "economies;",
    countries.income_group.value_counts().reindex(T.INCOME_ORDER).to_dict(),
)


## WDI indicators (2021–2025)

In [ ]:
INDICATORS = {
    "NY.GDP.PCAP.PP.KD": "gdp_pc_ppp",
    "IT.NET.BBND.P2": "fixed_bb_per100",
    "IT.NET.USER.ZS": "internet_users_pct",
    "IT.CEL.SETS.P2": "mobile_subs_per100",
    "IT.NET.SECR.P6": "secure_servers_per_m",
    "SP.URB.TOTL.IN.ZS": "urban_pct",
    "SP.POP.TOTL": "population",
}
rows = []
for code, name in INDICATORS.items():
    js = cached_json(
        f"https://api.worldbank.org/v2/country/all/indicator/{code}",
        WB / f"{code}.json",
        {"format": "json", "per_page": 20000, "date": "2021:2025"},
    )
    for r in js[1]:
        if r["value"] is not None:
            rows.append((r["countryiso3code"], name, int(r["date"]), float(r["value"])))
wdi = pd.DataFrame(rows, columns=["iso3", "indicator", "year", "value"])
latest = wdi.sort_values("year").groupby(["iso3", "indicator"]).last().reset_index()
ind = latest.pivot(index="iso3", columns="indicator", values="value")
ind_year = latest.pivot(index="iso3", columns="indicator", values="year").add_suffix(
    "_year"
)
countries = countries.merge(ind, left_on="iso3", right_index=True, how="left").merge(
    ind_year, left_on="iso3", right_index=True, how="left"
)
countries[list(INDICATORS.values())].notna().sum().to_frame("economies with a value")

In [ ]:
latest.groupby("indicator").year.value_counts().unstack(fill_value=0)

## ISP market concentration from APNIC user estimates

We compute the HHI from operators' market shares. The U.S. DOJ/FTC 2023 merger guidelines treat markets above 1,800 as highly concentrated. Shares are renormalized to the users APNIC attributes to the listed ASes.

Grouping ASes by operator name is approximate: it merges ASes registered under the same organization name, but not affiliates registered under different names. In India, for example, Bharti Airtel's mobile and fixed ASes count as two operators. Operator-level and AS-level HHI still rank countries almost identically (Spearman ρ = 0.98), so this does not affect how we group countries by concentration.

In [ ]:
ap_path = AP / "aspop_all.csv"
if not ap_path.exists():
    r = requests.get("https://stats.labs.apnic.net/cgi-bin/aspop?f=c", timeout=300)
    r.raise_for_status()
    ap_path.write_text(r.text)
ap = pd.read_csv(ap_path, skiprows=1, keep_default_na=False, na_values=[""])
ap.columns = [
    "rank",
    "asn",
    "as_name",
    "country_code",
    "users",
    "pct_country",
    "pct_internet",
    "samples",
]
print(ap_path.read_text().splitlines()[0])

LEGAL = r"\b(ltd|limited|inc|llc|l l c|plc|corp|corporation|co|company|sa|s a|sas|spa|srl|sarl|ag|gmbh|bv|nv|ab|as|asa|oy|oyj|jsc|pjsc|ojsc|cjsc|llp|pty|pvt|private|public|tbk|bhd|sdn|kft|zrt|doo|d o o|ad|ltda|eireli|ooo|tov|sp z o o|s r l|s p a|b v|n v)\b"


def operator(as_name):
    org = as_name.split(" - ", 1)[1] if " - " in as_name else as_name
    org = T.norm_name(org)
    org = re.sub(LEGAL, " ", org)
    return re.sub(r"\s+", " ", org).strip() or T.norm_name(as_name)


ap["as_name"] = ap.as_name.fillna(ap.asn)  # a few ASes have no name
ap["operator"] = ap.as_name.map(operator)
op = ap.groupby(["country_code", "operator"]).users.sum().reset_index()
op["share"] = op.users / op.groupby("country_code").users.transform("sum")
hhi = op.groupby("country_code").agg(
    isp_hhi=("share", lambda s: 10_000 * (s**2).sum()),
    isp_top1_share=("share", "max"),
    isp_operators=("share", "size"),
    apnic_users=("users", "sum"),
)
hhi["isp_effective_n"] = 10_000 / hhi.isp_hhi
asn_level = (
    ap.assign(s=ap.users / ap.groupby("country_code").users.transform("sum"))
    .groupby("country_code")
    .s.apply(lambda s: 10_000 * (s**2).sum())
)
hhi["isp_hhi_asn_level"] = asn_level
print(
    f"{len(hhi)} countries; grouping ASes by operator raises HHI for "
    f"{(hhi.isp_hhi > hhi.isp_hhi_asn_level + 1).mean():.0%} of them "
    f"(median +{(hhi.isp_hhi - hhi.isp_hhi_asn_level).median():.0f} points); "
    f"Spearman(operator-level, AS-level) = {spearmanr(hhi.isp_hhi, hhi.isp_hhi_asn_level).statistic:.3f}"
)
hhi.describe().round(2).T

In [ ]:
# Spot-check a few markets
for cc in ["US", "IN", "KE", "DE", "BR"]:
    top = op[op.country_code == cc].nlargest(4, "share")
    print(
        cc,
        f"HHI {hhi.loc[cc, 'isp_hhi']:.0f}:",
        "; ".join(f"{o} {s:.0%}" for o, s in zip(top.operator, top.share)),
    )


## M-Lab test-share HHI, for comparison

In [ ]:
asn = T.load_level(
    "country_asn", months=pd.date_range("2025-01-01", "2025-12-01", freq="MS")
)
tshare = asn.groupby(["country_code", "asn"]).n_down.sum().reset_index()
tshare["s"] = tshare.n_down / tshare.groupby("country_code").n_down.transform("sum")
mlab_hhi = (
    tshare.groupby("country_code")
    .s.apply(lambda s: 10_000 * (s**2).sum())
    .rename("mlab_test_hhi")
)

both = hhi.join(mlab_hhi, how="inner")
rho = spearmanr(both.isp_hhi_asn_level, both.mlab_test_hhi).statistic
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.scatter(
    both.isp_hhi_asn_level,
    both.mlab_test_hhi,
    s=14,
    color=T.SERIES[0],
    alpha=0.7,
    edgecolor=T.SURFACE,
    linewidth=0.6,
)
ax.plot([100, 10_000], [100, 10_000], color=T.AXIS, lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("HHI from APNIC user estimates (AS level)")
ax.set_ylabel("HHI from M-Lab test counts (AS level)")
ax.set_title(f"Two concentration proxies, {len(both)} countries\nSpearman ρ = {rho:.2f} (line: equality)")
T.savefig(fig, "0_hhi_apnic_vs_mlab")
print(
    f"M-Lab test HHI exceeds APNIC user HHI for "
    f"{(both.mlab_test_hhi > both.isp_hhi_asn_level).mean():.0%} of countries"
)

The two proxies rank countries similarly (Spearman ρ = 0.87), and for about one country in 20 they differ by more than a factor of two. Test-count HHI reflects who runs speed tests, and small countries have few tests, so we use the APNIC operator-level HHI as the market-concentration variable from here on.

## 7 · Assemble and save

We assemble one row per country in the M-Lab data, with its 2025 test counts, World Bank income group and indicators, and both HHI measures.

*Note:* Namibia's country code is `NA`, which pandas reads as a missing value by default. Read this table with `pd.read_csv(..., keep_default_na=False, na_values=[""])`.

In [ ]:
def country_name(cc):
    c = pycountry.countries.get(alpha_2=cc)
    return getattr(c, "common_name", None) or (c.name if c else cc)


C = (
    df[df.month.dt.year == 2025]
    .groupby("country_code")
    .agg(n_down=("n_down", "sum"), n_up=("n_up", "sum"), months=("month", "nunique"))
    .reset_index()
)
C.insert(1, "country", C.country_code.map(country_name))

ctx = (
    C.merge(countries.drop(columns=["wb_name"]), on="country_code", how="left")
    .merge(hhi.reset_index(), on="country_code", how="left")
    .merge(mlab_hhi.reset_index(), on="country_code", how="left")
)
ctx["log10_gdp_pc"] = np.log10(ctx.gdp_pc_ppp)
ctx["tests_per_1000_users"] = (
    1000 * ctx.n_down / (ctx.population * ctx.internet_users_pct / 100)
)
ctx["hhi_tercile"] = pd.qcut(ctx.isp_hhi, 3, labels=["Low HHI", "Mid HHI", "High HHI"])

print(
    f"{len(ctx)} M-Lab countries; with income group: {ctx.income_group.notna().sum()}, "
    f"GDP: {ctx.gdp_pc_ppp.notna().sum()}, fixed BB: {ctx.fixed_bb_per100.notna().sum()}, "
    f"internet users: {ctx.internet_users_pct.notna().sum()}, ISP HHI: {ctx.isp_hhi.notna().sum()}"
)
print("HHI tercile cut points:", ctx.isp_hhi.quantile([1 / 3, 2 / 3]).round(0).tolist())
ctx.to_csv(T.TAB / "country_context.csv", index=False)

pd.crosstab(ctx.income_group, ctx.hhi_tercile).reindex(T.INCOME_ORDER)

In [ ]:
print(
    f"Spearman(ISP HHI, log GDP per capita) = "
    f"{spearmanr(ctx.isp_hhi, ctx.log10_gdp_pc, nan_policy='omit').statistic:.2f}"
)

Concentration and income are only weakly related (ρ = −0.16): middle- and high-income markets are spread across all three HHI segments. Low-income markets, however, are rarely in the least concentrated tercile (2 of 25), so comparisons across HHI terciles are best made within income groups.